# Configuración e imports

In [ ]:
import gzip
import re
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('../../../src/')
import config
from Eigenvalues import D1_neumann_centered, D2_neumann_centered
from Utilities import *

#plt.rcParams["figure.dpi"] = 300
A = 0.638
# Cambia solo esta ruta para otros valores de A
BASE_RESTART_PATH = Path(
    "../../../scripts/Results/Blow_up/As/"
    f"blow_up_N_160000_Nt_64221_sigma_0.1_second/blow_up_A_{A}/restart_runs"
) # second

TARGET_INF = np.sqrt(2) / 2
TOL_INF = 5e-3

# w_<tiempo>.npy.gz
W_RE = re.compile(r"^w_([0-9]+(?:\.[0-9]+)?(?:e[+\-]?[0-9]+)?)$", re.I)

#  Utilidades de carga (.npy.gz + nombres exactos)

In [ ]:
def _strip_ext(name: str) -> str:
    s = name.lower()
    for ext in (".npy.gz", ".npy"):
        if s.endswith(ext):
            return s[: -len(ext)]
    return Path(s).stem

def _is_npy_or_npy_gz(p: Path) -> bool:
    n = p.name.lower()
    return n.endswith(".npy") or n.endswith(".npy.gz")

def _maybe_complex(arr):
    arr = np.asarray(arr)
    if np.iscomplexobj(arr):
        return np.squeeze(arr)
    if arr.ndim >= 1 and arr.shape[-1] == 2:
        return np.squeeze(arr[..., 0] + 1j * arr[..., 1])
    if arr.ndim == 2 and arr.shape[0] == 2 and arr.shape[1] != 2:
        return np.squeeze(arr[0] + 1j * arr[1])
    return np.squeeze(arr)

def load_npy_auto(path: Path):
    if path.name.lower().endswith(".npy.gz"):
        with gzip.open(path, "rb") as f:
            arr = np.load(f, allow_pickle=True)
    elif path.name.lower().endswith(".npy"):
        arr = np.load(path, allow_pickle=True)
    else:
        raise ValueError(f"No soportado: {path}")
    return _maybe_complex(arr)

def iter_data_files(run_dir: Path):
    return sorted([p for p in run_dir.rglob("*") if p.is_file() and _is_npy_or_npy_gz(p)])

def find_exact_named_files(run_dir: Path, base_name: str):
    base_name = base_name.lower()
    out = []
    for p in iter_data_files(run_dir):
        if _strip_ext(p.name) == base_name:
            out.append(p)
    return sorted(out)

def get_w_files(run_dir: Path):
    out = []
    for p in iter_data_files(run_dir):
        m = W_RE.match(_strip_ext(p.name))
        if m:
            out.append((float(m.group(1)), p))
    out.sort(key=lambda z: z[0])
    return out

def get_run_dirs(restart_dir: Path):
    if not restart_dir.exists():
        raise FileNotFoundError(f"No existe la ruta: {restart_dir}")
    dirs = sorted([d for d in restart_dir.iterdir() if d.is_dir()])
    if len(dirs) < 2:
        warnings.warn(f"Se esperaban al menos 2 subcarpetas y se encontraron {len(dirs)}")
    return dirs

def _is_no_refine_key(run_key: str) -> bool:
    key = run_key.lower()
    return ("no_refine" in key) or ("dx_1x_dt_1x" in key)

# Parseo de normas + derivadas

In [ ]:
def parse_norm_array(arr):
    a = np.asarray(arr)
    if a.ndim == 1:
        return None, np.real(a).ravel()

    if a.ndim == 2 and 2 in a.shape:
        if a.shape[1] == 2:
            return np.real(a[:, 0]).ravel(), np.real(a[:, 1]).ravel()
        if a.shape[0] == 2:
            return np.real(a[0, :]).ravel(), np.real(a[1, :]).ravel()

    return None, np.real(a).ravel()

def load_norm_series(paths):
    if not paths:
        return None, None

    t_parts, y_parts = [], []
    t_all_present = True

    for p in sorted(paths):
        try:
            t_i, y_i = parse_norm_array(load_npy_auto(p))
            if y_i is not None and len(y_i) > 0:
                y_parts.append(np.ravel(y_i))
                if t_i is None:
                    t_all_present = False
                else:
                    t_parts.append(np.ravel(t_i))
        except Exception as e:
            warnings.warn(f"No se pudo cargar {p.name}: {e}")

    if not y_parts:
        return None, None

    y = np.concatenate(y_parts)
    t = np.concatenate(t_parts) if (t_all_present and len(t_parts) == len(y_parts)) else None
    return t, y

def choose_space_array(paths):
    best = None
    for p in sorted(paths):
        try:
            a = np.ravel(load_npy_auto(p))
            if a.size == 0:
                continue
            if best is None or a.size > best.size:
                best = a
        except Exception as e:
            warnings.warn(f"No se pudo cargar {p.name}: {e}")
    return best


def compute_derivatives(u_last, x):
    if u_last is None:
        return None, None, None

    u = np.ravel(u_last)

    if x is None or len(np.ravel(x)) != len(u):
        x = np.arange(len(u), dtype=float)
    else:
        x = np.ravel(x).astype(float)

    if len(u) < 3:
        nanv = np.full_like(u, np.nan, dtype=complex)
        return nanv, nanv, x

    dx = np.diff(x)
    h = float(np.mean(dx)) if dx.size else 1.0
    if np.isclose(h, 0.0):
        raise ValueError("Paso espacial h=0; no se pueden construir D1/D2.")
    if dx.size and not np.allclose(dx, h, rtol=1e-8, atol=1e-12):
        warnings.warn("x no uniforme: D1/D2 de Eigenvalues usan h promedio.")

    D1 = D1_neumann_centered(len(u), h)
    D2 = D2_neumann_centered(len(u), h)

    ux = np.asarray(D1 @ u).ravel()
    uxx = np.asarray(D2 @ u).ravel()

    return ux, uxx, x


def _default_t(t, y, w_times):
    if y is None:
        return None, None
    y = np.ravel(y)
    if t is not None and len(np.ravel(t)) == len(y):
        return np.ravel(t), y
    if w_times is not None and len(w_times) >= len(y):
        return np.ravel(w_times[:len(y)]), y
    return np.arange(len(y), dtype=float), y

# Cargar una corrida (nombres exactos)

In [ ]:
def _effective_norm_inf_series(t_inf, norm_inf, zero_tol=1e-14):
    """
    Recorta la cola preasignada en cero de norm_inf.
    Conserva hasta el ultimo indice con |norm_inf| > zero_tol.
    """
    if norm_inf is None:
        return None, None

    y = np.ravel(np.real(norm_inf)).astype(float)
    if y.size == 0:
        return None, None

    if t_inf is None or len(np.ravel(t_inf)) != len(y):
        t = np.arange(len(y), dtype=float)
    else:
        t = np.ravel(t_inf).astype(float)

    nz = np.where(np.abs(y) > zero_tol)[0]
    if nz.size == 0:
        return None, None

    last = int(nz[-1])
    return t[: last + 1], y[: last + 1]


def _last_time_below_target_from_norm(t_eff, y_eff, target_inf):
    """
    En la serie efectiva de norm_inf, devuelve el ultimo tiempo con norm<=target.
    """
    if t_eff is None or y_eff is None or len(y_eff) == 0:
        return None, None

    idx = np.where(np.isfinite(y_eff) & (y_eff <= target_inf))[0]
    if idx.size == 0:
        return None, None

    k = int(idx[-1])
    return float(t_eff[k]), float(y_eff[k])


def _norm_time_step(t_eff):
    if t_eff is None or len(t_eff) < 2:
        return 0.0
    dt = np.diff(np.ravel(t_eff).astype(float))
    dt = dt[np.isfinite(dt)]
    dt = dt[dt > 0]
    if dt.size == 0:
        return 0.0
    return float(np.median(dt))


def _select_snapshot_with_criteria(
    w_pairs,
    target_inf,
    t_cut=None,
    t_cut_tol=0.0,
    norm_ref_val=None,
    norm_val_tol=1e-10,
):
    """
    Selecciona el ultimo snapshot que cumple max|u|<=target_inf y, si existe,
    t <= (t_cut + t_cut_tol). Si se entrega norm_ref_val, prioriza coincidencia
    de valor con el ultimo norm_inf no-cero.
    """
    if not w_pairs:
        return None

    selected = None
    selected_match = None
    physical_last = None
    selected_relaxed = None

    t_limit = None if t_cut is None else float(t_cut + t_cut_tol)

    for t_w, p_w in w_pairs:
        u_w = np.ravel(load_npy_auto(p_w))
        max_abs = float(np.max(np.abs(u_w))) if u_w.size else np.nan
        rec = (float(t_w), p_w, u_w, max_abs)
        physical_last = rec

        if np.isfinite(max_abs) and max_abs <= target_inf:
            selected_relaxed = rec
            if (t_limit is None) or (float(t_w) <= t_limit + 1e-12):
                selected = rec
                if (
                    norm_ref_val is not None
                    and np.isfinite(norm_ref_val)
                    and abs(max_abs - float(norm_ref_val)) <= norm_val_tol
                ):
                    selected_match = rec

    if selected_match is not None:
        mode = "target_norm_window_and_value"
        return selected_match, physical_last, mode

    if selected is not None:
        mode = "target_and_norm_window"
        return selected, physical_last, mode

    if selected_relaxed is not None:
        warnings.warn(
            "No hubo snapshot dentro de la ventana temporal de norm_inf; se usa ultimo snapshot con ||u||_inf<=TARGET_INF."
        )
        mode = "target_only_fallback"
        return selected_relaxed, physical_last, mode

    warnings.warn(
        "No hay snapshots con ||u||_inf<=TARGET_INF; se usa el ultimo snapshot disponible."
    )
    mode = "physical_last_fallback"
    return physical_last, physical_last, mode


def load_run(run_dir: Path):
    # Nombres exactos:
    # norm_inf.npy.gz, H1_norm.npy.gz, H2_norm.npy.gz, space.npy.gz, time.npy.gz, w_<t>.npy.gz
    f_inf = find_exact_named_files(run_dir, "norm_inf")
    f_h1  = find_exact_named_files(run_dir, "h1_norm")  # case-insensitive
    f_h2  = find_exact_named_files(run_dir, "h2_norm")  # case-insensitive
    f_space = find_exact_named_files(run_dir, "space")
    f_time = find_exact_named_files(run_dir, "time")
    w_pairs = get_w_files(run_dir)

    x = choose_space_array(f_space)
    t_file = choose_space_array(f_time)
    t_file = np.ravel(np.real(t_file)).astype(float) if t_file is not None else None

    t_inf_raw, norm_inf_raw = load_norm_series(f_inf)
    t_h1_raw, norm_h1 = load_norm_series(f_h1)
    t_h2_raw, norm_h2 = load_norm_series(f_h2)

    w_times = np.array([t for t, _ in w_pairs], dtype=float) if w_pairs else np.array([], dtype=float)

    # Tiempos para SELECCION: mantener escala usada para comparar con w_<t>
    t_inf_sel, norm_inf_raw = _default_t(t_inf_raw, norm_inf_raw, w_times)
    t_h1_sel, norm_h1 = _default_t(t_h1_raw, norm_h1, w_times)
    t_h2_sel, norm_h2 = _default_t(t_h2_raw, norm_h2, w_times)

    # Tiempos para REPORTE/GRAFICO: preferir archivo time cuando calza por longitud
    t_inf_plot = t_inf_sel
    t_h1_plot = t_h1_sel
    t_h2_plot = t_h2_sel

    if t_file is not None and norm_inf_raw is not None and len(t_file) >= len(np.ravel(norm_inf_raw)):
        t_inf_plot = t_file[: len(np.ravel(norm_inf_raw))]
    if t_file is not None and norm_h1 is not None and len(t_file) >= len(np.ravel(norm_h1)):
        t_h1_plot = t_file[: len(np.ravel(norm_h1))]
    if t_file is not None and norm_h2 is not None and len(t_file) >= len(np.ravel(norm_h2)):
        t_h2_plot = t_file[: len(np.ravel(norm_h2))]

    t_inf_eff_sel, norm_inf_eff = _effective_norm_inf_series(t_inf_sel, norm_inf_raw)
    t_cut, norm_at_t_cut = _last_time_below_target_from_norm(t_inf_eff_sel, norm_inf_eff, TARGET_INF)
    dt_norm = _norm_time_step(t_inf_eff_sel)

    u_last, t_last, u_file_last = None, None, None
    u_last_inf = None
    t_last_physical, u_file_last_physical, u_last_physical_inf = None, None, None
    selected_mode = "no_w_files"

    norm_val_tol = 1e-10

    if w_pairs:
        selected, physical_last, selected_mode = _select_snapshot_with_criteria(
            w_pairs,
            TARGET_INF,
            t_cut=t_cut,
            t_cut_tol=dt_norm,
            norm_ref_val=norm_at_t_cut,
            norm_val_tol=norm_val_tol,
        )
        if selected is not None:
            t_last, u_file_last, u_last, u_last_inf = selected
        if physical_last is not None:
            t_last_physical, u_file_last_physical, _, u_last_physical_inf = physical_last

    ux_last, uxx_last, x_plot = compute_derivatives(u_last, x)

    selected_norm_diff = (
        abs(float(u_last_inf) - float(norm_at_t_cut))
        if (u_last_inf is not None and norm_at_t_cut is not None)
        else None
    )

    return {
        "label": run_dir.name,
        "run_dir": run_dir,
        "files": {
            "norm_inf": f_inf,
            "norm_h1": f_h1,
            "norm_h2": f_h2,
            "space": f_space,
            "time": f_time,
            "w_count": len(w_pairs),
            "w_last": u_file_last,
            "w_last_physical": u_file_last_physical,
        },
        "x_plot": np.ravel(x_plot) if x_plot is not None else None,
        "u_last": np.ravel(u_last) if u_last is not None else None,
        "ux_last": np.ravel(ux_last) if ux_last is not None else None,
        "uxx_last": np.ravel(uxx_last) if uxx_last is not None else None,
        "t_last_w": t_last,
        "u_last_inf": u_last_inf,
        "t_last_w_physical": t_last_physical,
        "u_last_physical_inf": u_last_physical_inf,
        "selected_mode": selected_mode,
        "selected_norm_diff": selected_norm_diff,
        "selected_norm_val_tol": norm_val_tol,
        "norm_inf_t": t_inf_plot,
        "norm_inf_t_sel": t_inf_sel,
        "norm_inf": norm_inf_raw,
        "norm_inf_t_eff": _effective_norm_inf_series(t_inf_plot, norm_inf_raw)[0],
        "norm_inf_eff": norm_inf_eff,
        "norm_cut_t": t_cut,
        "norm_cut_val": norm_at_t_cut,
        "norm_dt": dt_norm,
        "norm_h1_t": t_h1_plot,
        "norm_h2_t": t_h2_plot,
        "norm_h1": norm_h1,
        "norm_h2": norm_h2,
    }

#  Ejecutar carga + chequeo

In [ ]:
restart_dir = BASE_RESTART_PATH.expanduser().resolve()
run_dirs = get_run_dirs(restart_dir)
runs = {d.name: load_run(d) for d in run_dirs}

print(f"Ruta: {restart_dir}")
print(f"Subcarpetas detectadas: {list(runs.keys())}\n")

for label, d in runs.items():
    print(f"--- {label} ---")
    print("norm_inf:", [p.name for p in d["files"]["norm_inf"]])
    print("H1_norm :", [p.name for p in d["files"]["norm_h1"]])
    print("H2_norm :", [p.name for p in d["files"]["norm_h2"]])
    print("space   :", [p.name for p in d["files"]["space"]])
    print("w_count :", d["files"]["w_count"])
    print("selected_mode:", d["selected_mode"])
    print("w_selected:", d["files"]["w_last"].name if d["files"]["w_last"] else None)
    print("w_last_physical:", d["files"]["w_last_physical"].name if d["files"]["w_last_physical"] else None)

    t_cut = d.get("norm_cut_t")
    y_cut = d.get("norm_cut_val")
    dt_norm = d.get("norm_dt", 0.0)
    if t_cut is None:
        print("norm_inf efectivo: sin punto <= TARGET_INF")
    else:
        print(
            f"norm_inf efectivo ultimo <= target: t={t_cut:.8f}, norm={y_cut:.8f}, "
            f"ventana=[-inf, {t_cut + dt_norm:.8f}]"
        )

    if d.get("u_last_inf") is not None:
        v = float(d["u_last_inf"])
        print(
            f"snapshot usado: max|u|={v:.8f}, <=target={v <= TARGET_INF}, "
            f"diff={abs(v-TARGET_INF):.3e}"
        )
    else:
        print("No se selecciono snapshot de w")

    sel_diff = d.get("selected_norm_diff")
    sel_tol = d.get("selected_norm_val_tol")
    if sel_diff is not None and sel_tol is not None:
        print(
            f"match con ultimo norm_inf no-cero: |diff|={sel_diff:.3e}, "
            f"tol={sel_tol:.1e}, ok={sel_diff <= sel_tol}"
        )

    t_eff = d.get("norm_inf_t_eff")
    y_eff = d.get("norm_inf_eff")
    if t_eff is not None and y_eff is not None and len(y_eff) > 0:
        print(f"norm_inf efectivo ultimo valor: t={float(t_eff[-1]):.8f}, norm={float(y_eff[-1]):.8f}")

    print()

#  Funciones de plot (UNA por tipo)

In [ ]:
# ...existing code...

def _iter_runs(runs_dict, run_order=None):
    if run_order is None:
        return list(runs_dict.items())
    keys = [k for k in run_order if k in runs_dict]
    keys += [k for k in runs_dict.keys() if k not in keys]
    return [(k, runs_dict[k]) for k in keys]

def _label_for(run_key, run_data, legend_labels=None):
    if legend_labels is None:
        return run_data.get("label", run_key)
    return legend_labels.get(run_key, run_data.get("label", run_key))

def _get_xy(d, field_key):
    y = d.get(field_key, None)
    if y is None:
        return None, None
    y = np.ravel(y)
    x = d.get("x_plot", None)
    if x is None or len(np.ravel(x)) != len(y):
        x = np.arange(len(y))
    else:
        x = np.ravel(x)
    return x, y

def _resolve_limits(L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None):
    if xlim is None and L is not None:
        xlim = (-L, L)
    if ylim is None and Ly_min is not None and Ly_max is not None:
        ylim = (Ly_min, Ly_max)
    return xlim, ylim

def _apply_limits(xlim=None, ylim=None):
    if xlim is not None:
        plt.xlim(*xlim)
    if ylim is not None:
        plt.ylim(*ylim)

def _plot_field_real(runs_dict, field_key, legend_labels=None, run_order=None, ms=0, L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None, ylabel="\Psi"):
    xlim, ylim = _resolve_limits(L=L, Ly_min=Ly_min, Ly_max=Ly_max, xlim=xlim, ylim=ylim)
    plt.figure()
    for run_key, d in _iter_runs(runs_dict, run_order):
        x, y = _get_xy(d, field_key)
        if y is None:
            continue
        plt.plot(x, np.real(y), label=_label_for(run_key, d, legend_labels), markersize=ms)
    plt.xlabel(r"$x$")
    plt.ylabel(r"$\mathrm{Re}$"+rf"$({ylabel})$", rotation=0, labelpad=25)
    _apply_limits(xlim=xlim, ylim=ylim)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

def _plot_field_imag(runs_dict, field_key, legend_labels=None, run_order=None, ms=0, L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None, ylabel="\Psi"):
    xlim, ylim = _resolve_limits(L=L, Ly_min=Ly_min, Ly_max=Ly_max, xlim=xlim, ylim=ylim)
    plt.figure()
    for run_key, d in _iter_runs(runs_dict, run_order):
        x, y = _get_xy(d, field_key)
        if y is None:
            continue
        plt.plot(x, np.imag(y), label=_label_for(run_key, d, legend_labels), markersize=ms)
    plt.xlabel(r"$x$")
    plt.ylabel(r"$\mathrm{Im}$"+rf"$({ylabel})$", rotation=0, labelpad=25)
    _apply_limits(xlim=xlim, ylim=ylim)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

# Funciones separadas: REAL / IMAG
def plot_u_last_real(runs_dict, legend_labels=None, run_order=None, ms=0, L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None):
    _plot_field_real(runs_dict, "u_last", legend_labels, run_order, ms, L, Ly_min, Ly_max, xlim, ylim)

def plot_u_last_imag(runs_dict, legend_labels=None, run_order=None, ms=0, L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None):
    _plot_field_imag(runs_dict, "u_last", legend_labels, run_order, ms, L, Ly_min, Ly_max, xlim, ylim)

def plot_ux_last_real(runs_dict, legend_labels=None, run_order=None, ms=0, L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None, ylabel=None):
    _plot_field_real(runs_dict, "ux_last", legend_labels, run_order, ms, L, Ly_min, Ly_max, xlim, ylim, ylabel)

def plot_ux_last_imag(runs_dict, legend_labels=None, run_order=None, ms=0, L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None, ylabel=None):
    _plot_field_imag(runs_dict, "ux_last", legend_labels, run_order, ms, L, Ly_min, Ly_max, xlim, ylim, ylabel)

def plot_uxx_last_real(runs_dict, legend_labels=None, run_order=None, ms=0, L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None, ylabel=None):
    _plot_field_real(runs_dict, "uxx_last", legend_labels, run_order, ms, L, Ly_min, Ly_max, xlim, ylim, ylabel)

def plot_uxx_last_imag(runs_dict, legend_labels=None, run_order=None, ms=0, L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None, ylabel=None):
    _plot_field_imag(runs_dict, "uxx_last", legend_labels, run_order, ms, L, Ly_min, Ly_max, xlim, ylim, ylabel)

# Wrappers (compatibles con llamadas viejas) con límites distintos real/imag
def plot_u_last(runs_dict, legend_labels=None, run_order=None, ms=0,
                L=None, Ly_min=None, Ly_max=None,
                L_real=None, Ly_min_real=None, Ly_max_real=None, xlim_real=None, ylim_real=None,
                L_imag=None, Ly_min_imag=None, Ly_max_imag=None, xlim_imag=None, ylim_imag=None):
    L_real = L if L_real is None else L_real
    Ly_min_real = Ly_min if Ly_min_real is None else Ly_min_real
    Ly_max_real = Ly_max if Ly_max_real is None else Ly_max_real
    L_imag = L if L_imag is None else L_imag
    Ly_min_imag = Ly_min if Ly_min_imag is None else Ly_min_imag
    Ly_max_imag = Ly_max if Ly_max_imag is None else Ly_max_imag

    plot_u_last_real(runs_dict, legend_labels, run_order, ms, L_real, Ly_min_real, Ly_max_real, xlim_real, ylim_real)
    plot_u_last_imag(runs_dict, legend_labels, run_order, ms, L_imag, Ly_min_imag, Ly_max_imag, xlim_imag, ylim_imag)




def plot_u_last_mod(runs_dict, legend_labels=None, run_order=None, ms=0,
                    L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None,
                    save_path=None, ms_zoom=None, refined_color_index=None, **kwargs):
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

    def _curve_plot_kwargs(run_key):
        curve_kwargs = dict(kwargs)
        if refined_color_index is not None and not _is_no_refine_key(run_key):
            colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
            curve_kwargs["color"] = colors[int(refined_color_index) % len(colors)]
        return curve_kwargs

    zoom_xlim, zoom_ylim = _resolve_limits(
        L=L, Ly_min=Ly_min, Ly_max=Ly_max, xlim=xlim, ylim=ylim
    )

    fig, ax = plt.subplots()
    plotted = []
    legend_handles, legend_texts = [], []
    x0, x1 = None, None

    for run_key, d in _iter_runs(runs_dict, run_order):
        x, y = _get_xy(d, "u_last")
        if y is None:
            continue
        yy = np.abs(y)
        label = _label_for(run_key, d, legend_labels)
        curve_kwargs = _curve_plot_kwargs(run_key)
        line, = ax.plot(x, yy, label=label, markersize=ms, **curve_kwargs)
        plotted.append((x, yy, label, line, curve_kwargs))
        legend_handles.append(line)
        legend_texts.append(label)
        xmin_i, xmax_i = np.min(x), np.max(x)
        x0 = xmin_i if x0 is None else min(x0, xmin_i)
        x1 = xmax_i if x1 is None else max(x1, xmax_i)

    target_label = r"$\sqrt{2}/2$"
    if x0 is not None and x1 is not None:
        target_handle = ax.hlines(TARGET_INF, x0, x1, colors="grey", linestyles="dashed", label=target_label)
        legend_handles.append(target_handle)
        legend_texts.append(target_label)

    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$|\Psi|$", rotation=0, labelpad=10, fontsize=11)
    plt.tight_layout()

    if plotted and (zoom_xlim is not None or zoom_ylim is not None):
        axins = inset_axes(ax, width="40%", height="40%", loc="center left", borderpad=1.2)
        ms_zoom_use = ms if ms_zoom is None else ms_zoom
        inset_handles, inset_labels = [], []

        for x, yy, label, _, curve_kwargs in plotted:
            inset_line, = axins.plot(x, yy, markersize=ms_zoom_use, **curve_kwargs)
            inset_handles.append(inset_line)
            inset_labels.append(label)

        if x0 is not None and x1 is not None:
            inset_target = axins.hlines(TARGET_INF, x0, x1, colors="grey", linestyles="dashed")
            inset_handles.append(inset_target)
            inset_labels.append(target_label)

        legend_handles, legend_texts = inset_handles, inset_labels

        if zoom_xlim is not None:
            axins.set_xlim(*zoom_xlim)
        if zoom_ylim is not None:
            axins.set_ylim(*zoom_ylim)

        axins.set_xlabel("")
        axins.set_ylabel("")
        axins.tick_params(labelbottom=False, labelleft=False)
        axins.grid(alpha=0.2)

        x_min_zoom, x_max_zoom = axins.get_xlim()
        y_min_zoom, y_max_zoom = axins.get_ylim()

        fs_txt = 6       # números más pequeños

        bbox_txt = dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.4)
        axins.text(0.02, 0.98, f"{y_max_zoom:.6g}", transform=axins.transAxes,
                ha="left", va="top", fontsize=fs_txt, clip_on=True, bbox=bbox_txt)
        axins.text(0.02, 0.02, f"{y_min_zoom:.6g}", transform=axins.transAxes,
                ha="left", va="bottom", fontsize=fs_txt, clip_on=True, bbox=bbox_txt)
        axins.text(0.02, -0.07, f"{x_min_zoom:.6g}", transform=axins.transAxes,
                ha="left", va="top", fontsize=fs_txt, clip_on=False)
        axins.text(0.98, -0.07, f"{x_max_zoom:.6g}", transform=axins.transAxes,
                ha="right", va="top", fontsize=fs_txt, clip_on=False)

        mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.4", lw=0.5, alpha=0.5)

    if legend_handles:
        ax.legend(legend_handles, legend_texts, fontsize=10, loc="center right")

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_u_x_last_mod(runs_dict, legend_labels=None, run_order=None, ms=0,
                      L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None,
                      save_path=None, ms_zoom=None, refined_color_index=None, **kwargs):
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

    def _curve_plot_kwargs(run_key):
        curve_kwargs = dict(kwargs)
        if refined_color_index is not None and not _is_no_refine_key(run_key):
            colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
            curve_kwargs["color"] = colors[int(refined_color_index) % len(colors)]
        return curve_kwargs

    zoom_xlim, zoom_ylim = _resolve_limits(
        L=L, Ly_min=Ly_min, Ly_max=Ly_max, xlim=xlim, ylim=ylim
    )

    fig, ax = plt.subplots()
    plotted = []
    legend_handles, legend_texts = [], []

    for run_key, d in _iter_runs(runs_dict, run_order):
        x, y = _get_xy(d, "ux_last")
        if y is None:
            continue
        yy = np.abs(y)
        label = _label_for(run_key, d, legend_labels)
        curve_kwargs = _curve_plot_kwargs(run_key)
        line, = ax.plot(x, yy, label=label, markersize=ms, **curve_kwargs)
        plotted.append((x, yy, label, line, curve_kwargs))
        legend_handles.append(line)
        legend_texts.append(label)

    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$|D_{x}^h \Psi|$", rotation=0, labelpad=18, fontsize=11)
    plt.tight_layout()

    if plotted and (zoom_xlim is not None or zoom_ylim is not None):
        axins = inset_axes(ax, width="40%", height="40%", loc="center left", borderpad=1.2)
        ms_zoom_use = ms if ms_zoom is None else ms_zoom
        inset_handles, inset_labels = [], []

        for x, yy, label, _, curve_kwargs in plotted:
            inset_line, = axins.plot(x, yy, markersize=ms_zoom_use, **curve_kwargs)
            inset_handles.append(inset_line)
            inset_labels.append(label)

        legend_handles, legend_texts = inset_handles, inset_labels

        if zoom_xlim is not None:
            axins.set_xlim(*zoom_xlim)
        if zoom_ylim is not None:
            axins.set_ylim(*zoom_ylim)

        axins.set_xlabel("")
        axins.set_ylabel("")
        axins.tick_params(labelbottom=False, labelleft=False)
        axins.grid(alpha=0.2)

        x_min_zoom, x_max_zoom = axins.get_xlim()
        y_min_zoom, y_max_zoom = axins.get_ylim()

        fs_txt = 6       # números más pequeños

        bbox_txt = dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.4)
        axins.text(0.02, 0.98, f"{y_max_zoom:.6g}", transform=axins.transAxes,
                ha="left", va="top", fontsize=fs_txt, clip_on=True, bbox=bbox_txt)
        axins.text(0.02, 0.02, f"{y_min_zoom:.6g}", transform=axins.transAxes,
                ha="left", va="bottom", fontsize=fs_txt, clip_on=True, bbox=bbox_txt)
        axins.text(0.02, -0.07, f"{x_min_zoom:.6g}", transform=axins.transAxes,
                ha="left", va="top", fontsize=fs_txt, clip_on=False)
        axins.text(0.98, -0.07, f"{x_max_zoom:.6g}", transform=axins.transAxes,
                ha="right", va="top", fontsize=fs_txt, clip_on=False)

        mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.4", lw=0.5, alpha=0.5)

    if legend_handles:
        ax.legend(legend_handles, legend_texts, loc="center right", fontsize=10)

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_u_xx_last_mod(runs_dict, legend_labels=None, run_order=None, ms=0, LL=None, LLy_min=None, LLy_max=None,
                       L=None, Ly_min=None, Ly_max=None, xlim=None, ylim=None,
                       L2=None, Ly_min2=None, Ly_max2=None, xlim2=None, ylim2=None,
                       inset1_bbox=(-0.98, 0.08, 0.84, 0.84), inset2_bbox=(1.14, 0.08, 0.84, 0.84),
                       save_path=None, ms_zoom=None, ms_zoom2=None, refined_color_index=None, lw_big=None, **kwargs):
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

    def _curve_plot_kwargs(run_key):
        curve_kwargs = dict(kwargs)
        if refined_color_index is not None and not _is_no_refine_key(run_key):
            colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
            curve_kwargs["color"] = colors[int(refined_color_index) % len(colors)]
        return curve_kwargs

    zoom_xlim, zoom_ylim = _resolve_limits(
        L=L, Ly_min=Ly_min, Ly_max=Ly_max, xlim=xlim, ylim=ylim
    )
    zoom_xlim2, zoom_ylim2 = _resolve_limits(
        L=L2, Ly_min=Ly_min2, Ly_max=Ly_max2, xlim=xlim2, ylim=ylim2
    )

    fig, ax = plt.subplots(figsize=(9.5,2.5))
    plotted = []
    main_legend_handles, main_legend_texts = [], []
    inset_legend_handles, inset_legend_texts = [], []

    for run_key, d in _iter_runs(runs_dict, run_order):
        x, y = _get_xy(d, "uxx_last")
        if y is None:
            continue
        yy = np.abs(y)
        label = _label_for(run_key, d, legend_labels)
        curve_kwargs = _curve_plot_kwargs(run_key)
        line, = ax.plot(x, yy, label=label, markersize=ms, lw=lw_big, **curve_kwargs)
        plotted.append((x, yy, label, line, curve_kwargs))
        main_legend_handles.append(line)
        main_legend_texts.append(label)

    plt.ticklabel_format(axis='y', style='sci', scilimits=(4,4))
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$|D_{xx}^h \Psi|$", rotation=0, labelpad=-30, fontsize=11)

    if LL is not None:
        ax.set_xlim(-LL,LL)

    if LLy_min is not None and LLy_max is not None:
        ax.set_ylim(LLy_min, LLy_max)

    fig.subplots_adjust(left=0.38, right=0.62)

    def _add_mod_inset(inset_xlim, inset_ylim, bbox, ms_zoom_value, mark_locs):
        axins = inset_axes(
            ax,
            width="100%",
            height="100%",
            loc="lower left",
            bbox_to_anchor=bbox,
            bbox_transform=ax.transAxes,
            borderpad=0,
        )
        ms_zoom_use = ms if ms_zoom_value is None else ms_zoom_value

        inset_handles, inset_labels = [], []
        for x, yy, label, _, curve_kwargs in plotted:
            inset_line, = axins.plot(x, yy, markersize=ms_zoom_use, **curve_kwargs)
            inset_handles.append(inset_line)
            inset_labels.append(label)

        if inset_xlim is not None:
            axins.set_xlim(*inset_xlim)
        if inset_ylim is not None:
            axins.set_ylim(*inset_ylim)

        axins.set_xlabel("")
        axins.set_ylabel("")
        axins.tick_params(labelbottom=False, labelleft=False)
        axins.grid(alpha=0.2)

        x_min_zoom, x_max_zoom = axins.get_xlim()
        y_min_zoom, y_max_zoom = axins.get_ylim()

        fs_txt = 6       # números más pequeños

        def _fmt_plain(v):
            return f"{float(v):.6g}"

        y_abs_max = max(abs(float(y_min_zoom)), abs(float(y_max_zoom)))
        y_exp = int(np.floor(np.log10(y_abs_max)))-1 if y_abs_max > 0 else 0
        use_y_sci = y_exp >= 4
        y_scale = 10.0 ** y_exp if use_y_sci else 1.0
        y_factor = rf"$\times 10^{{{y_exp}}}$" if use_y_sci else None

        bbox_txt = dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.4)
        if y_factor is not None:
            axins.text(0.02, 1.07, y_factor, transform=axins.transAxes,
                    ha="left", va="top", fontsize=fs_txt, clip_on=False)
        axins.text(-0.02, 0.98, _fmt_plain(y_max_zoom / y_scale), transform=axins.transAxes,
                ha="right", va="top", fontsize=fs_txt, clip_on=False, bbox=bbox_txt)
        axins.text(-0.02, 0.02, _fmt_plain(y_min_zoom / y_scale), transform=axins.transAxes,
                ha="right", va="bottom", fontsize=fs_txt, clip_on=False, bbox=bbox_txt)
        axins.text(0.02, -0.07, _fmt_plain(x_min_zoom), transform=axins.transAxes,
                ha="left", va="top", fontsize=fs_txt, clip_on=False)
        axins.text(0.98, -0.07, _fmt_plain(x_max_zoom), transform=axins.transAxes,
                ha="right", va="top", fontsize=fs_txt, clip_on=False)

        mark_inset(ax, axins, loc1=mark_locs[0], loc2=mark_locs[1], fc="none", ec="0.4", lw=0.5, alpha=0.5)
        return inset_handles, inset_labels

    if plotted and (zoom_xlim is not None or zoom_ylim is not None):
        inset_legend_handles, inset_legend_texts = _add_mod_inset(zoom_xlim, zoom_ylim, bbox=inset1_bbox, ms_zoom_value=ms_zoom, mark_locs=(1, 3))

    if plotted and (zoom_xlim2 is not None or zoom_ylim2 is not None):
        # Preferir el segundo inset para la leyenda si existe, porque suele ser el zoom con marcadores visibles.
        inset_legend_handles, inset_legend_texts = _add_mod_inset(zoom_xlim2, zoom_ylim2, bbox=inset2_bbox, ms_zoom_value=ms_zoom2, mark_locs=(2, 4))

    if inset_legend_handles:
        ax.legend(inset_legend_handles, inset_legend_texts, fontsize=10, loc="center right")
    elif main_legend_handles:
        ax.legend(main_legend_handles, main_legend_texts, fontsize=10, loc="center right")

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()



def plot_ux_last(runs_dict, legend_labels=None, run_order=None, ms=0,
                 L=None, Ly_min=None, Ly_max=None,
                 L_real=None, Ly_min_real=None, Ly_max_real=None, xlim_real=None, ylim_real=None,
                 L_imag=None, Ly_min_imag=None, Ly_max_imag=None, xlim_imag=None, ylim_imag=None,
                 ylabel=None):
    L_real = L if L_real is None else L_real
    Ly_min_real = Ly_min if Ly_min_real is None else Ly_min_real
    Ly_max_real = Ly_max if Ly_max_real is None else Ly_max_real
    L_imag = L if L_imag is None else L_imag
    Ly_min_imag = Ly_min if Ly_min_imag is None else Ly_min_imag
    Ly_max_imag = Ly_max if Ly_max_imag is None else Ly_max_imag

    plot_ux_last_real(runs_dict, legend_labels, run_order, ms, L_real, Ly_min_real, Ly_max_real, xlim_real, ylim_real, ylabel)
    plot_ux_last_imag(runs_dict, legend_labels, run_order, ms, L_imag, Ly_min_imag, Ly_max_imag, xlim_imag, ylim_imag, ylabel)


def plot_uxx_last(runs_dict, legend_labels=None, run_order=None, ms=0,
                  L=None, Ly_min=None, Ly_max=None,
                  L_real=None, Ly_min_real=None, Ly_max_real=None, xlim_real=None, ylim_real=None,
                  L_imag=None, Ly_min_imag=None, Ly_max_imag=None, xlim_imag=None, ylim_imag=None, ylabel=None):
    L_real = L if L_real is None else L_real
    Ly_min_real = Ly_min if Ly_min_real is None else Ly_min_real
    Ly_max_real = Ly_max if Ly_max_real is None else Ly_max_real
    L_imag = L if L_imag is None else L_imag
    Ly_min_imag = Ly_min if Ly_min_imag is None else Ly_min_imag
    Ly_max_imag = Ly_max if Ly_max_imag is None else Ly_max_imag

    plot_uxx_last_real(runs_dict, legend_labels, run_order, ms, L_real, Ly_min_real, Ly_max_real, xlim_real, ylim_real, ylabel)
    plot_uxx_last_imag(runs_dict, legend_labels, run_order, ms, L_imag, Ly_min_imag, Ly_max_imag, xlim_imag, ylim_imag, ylabel)

def _plot_norm(runs_dict, t_key, y_key, title, ylabel, legend_labels=None, run_order=None, ms=0):
    plt.figure()
    for run_key, d in _iter_runs(runs_dict, run_order):
        y = d.get(y_key, None)
        t = d.get(t_key, None)
        if y is None:
            continue
        y = np.ravel(np.real(y))
        if t is None or len(np.ravel(t)) != len(y):
            t = np.arange(len(y))
        else:
            t = np.ravel(t)
        plt.plot(t, y, lw=1.8, label=_label_for(run_key, d, legend_labels), markersize=ms)
    plt.title(title)
    plt.xlabel("t")
    plt.ylabel(ylabel, rotation=0, labelpad=25)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()


def plot_norm_inf(runs_dict, legend_labels=None, run_order=None, ms=0):
    _plot_norm(runs_dict, "norm_inf_t", "norm_inf", r"$\|u\|_\infty$", r"$\|u\|_\infty$", legend_labels, run_order, ms)

def plot_norm_h1(runs_dict, legend_labels=None, run_order=None, ms=0):
    _plot_norm(runs_dict, "norm_h1_t", "norm_h1", r"$\|u\|_{H^1}$", r"$\|u\|_{H^1}$", legend_labels, run_order, ms)

def plot_norm_h2(runs_dict, legend_labels=None, run_order=None, ms=0):
    _plot_norm(runs_dict, "norm_h2_t", "norm_h2", r"$\|u\|_{H^2}$", r"$\|u\|_{H^2}$", legend_labels, run_order, ms)

# Graficar todo

In [ ]:
run_keys = list(runs.keys())

LEGEND_LABELS = {}
if len(run_keys) >= 1:
    LEGEND_LABELS[run_keys[0]] = "Refined"
if len(run_keys) >= 2:
    LEGEND_LABELS[run_keys[1]] = "Coarse"

# opcional: forzar orden en la leyenda
RUN_ORDER = run_keys[::-1]

In [ ]:
save_path = "Img/Heaven/Refined_comparisons/"
plot_u_last_mod(runs, legend_labels=LEGEND_LABELS, run_order=RUN_ORDER, ms=0, ms_zoom=1.5, 
                L=0.0001, Ly_min=0.70702, Ly_max=0.7071105, lw=1, save_path=save_path+"u_zoom.pdf",
                refined_color_index=2)

# A = 0.644
#plot_u_last_mod(runs, legend_labels=LEGEND_LABELS, run_order=RUN_ORDER, ms=0, ms_zoom=1.2, 
#                L=0.00008, Ly_min=0.70708, Ly_max=0.7071105, lw=1)

In [ ]:
plot_u_xx_last_mod(runs, legend_labels=LEGEND_LABELS, run_order=RUN_ORDER, ms=0, ms_zoom=0,
                L=0.8, Ly_min=-10, Ly_max=100, xlim2=(-0.0001, 0.0001), ylim2=(10_000, 101_000), ms_zoom2=1.5,
                lw_big=1, save_path=save_path+"u_xx_zoom.pdf", refined_color_index=2, LL=1)

In [ ]:
plot_u_x_last_mod(runs, legend_labels=LEGEND_LABELS, run_order=RUN_ORDER, ms=0, ms_zoom=1.5, 
                L=0.0001, Ly_min=-0.1, Ly_max=1.5, lw=1, 
                save_path=save_path+"u_x_zoom.pdf", refined_color_index=2)

In [ ]:
plot_ux_last(runs, legend_labels=LEGEND_LABELS, run_order=RUN_ORDER, ms=0, ylabel=r"\partial_x \Psi")

In [ ]:
def get_max_abs_arrays(runs_dict, run_order=None, with_labels=False):
    """
    Retorna dos arreglos (uno por subcarpeta/run) con:
    [max(|u|), max(|u_x|), max(|u_xx|)].

    Si with_labels=True, retorna:
    ((run_key_1, arr1), (run_key_2, arr2))
    """
    def _max_abs_or_nan(a):
        if a is None:
            return np.nan
        a = np.ravel(a)
        if a.size == 0:
            return np.nan
        return float(np.nanmax(np.abs(a)))

    vals = []
    keys = []

    for run_key, d in _iter_runs(runs_dict, run_order):
        arr = np.array([
            _max_abs_or_nan(d.get("u_last")),
            _max_abs_or_nan(d.get("ux_last")),
            _max_abs_or_nan(d.get("uxx_last")),
        ], dtype=float)
        vals.append(arr)
        keys.append(run_key)

    if len(vals) != 2:
        raise ValueError(f"Se esperaban 2 subcarpetas y se encontraron {len(vals)}: {keys}")

    if with_labels:
        return (keys[0], vals[0]), (keys[1], vals[1])

    return vals[0], vals[1]


In [ ]:
(run1_key, max_run_1), (run2_key, max_run_2) = get_max_abs_arrays(
    runs, run_order=RUN_ORDER, with_labels=True
)

delta_u_run_1 = TARGET_INF - max_run_1[0]
delta_u_run_2 = TARGET_INF - max_run_2[0]

print("Refined -> [target-max|u|, max|ux|, max|uxx|]:")
print("[")
print(f"  {sci_notation(float(delta_u_run_2), n_decimals=4)},")
print(f"  {float(max_run_2[1])},")
print(f"  {sci_notation(float(max_run_2[2]), n_decimals=4)}")
print("]")

print("A:", format(A, ".17f"))
print("TARGET_INF:", format(float(TARGET_INF), ".17f"))
print("No refined -> [target-max|u|, max|ux|, max|uxx|]:")
print("[")
print(f"  {sci_notation(float(delta_u_run_1), n_decimals=4)},")
print(f"  {float(max_run_1[1])},")
print(f"  {sci_notation(float(max_run_1[2]), n_decimals=4)}")
print("]")

In [ ]:
for run_key, d in runs.items():
    label = "Not refined" if _is_no_refine_key(run_key) else "Refined"

    ninf = d.get("norm_inf")
    t_inf = d.get("norm_inf_t")
    nh2 = d.get("norm_h2")
    t_h2 = d.get("norm_h2_t")

    if ninf is None or len(np.ravel(ninf)) == 0:
        print(f"--- {label} ---")
        print("norm_inf vacio")
        print()
        continue

    y_inf = np.ravel(np.real(ninf)).astype(float)
    has_time_inf = t_inf is not None and len(np.ravel(t_inf)) == len(y_inf)
    if has_time_inf:
        tt_inf = np.ravel(t_inf).astype(float)
    else:
        tt_inf = np.arange(len(y_inf), dtype=float)

    nz = np.where(np.abs(y_inf) > 1e-14)[0]
    if nz.size == 0:
        print(f"--- {label} ---")
        print("norm_inf no tiene valores no-cero")
        print()
        continue

    i_last = int(nz[-1])
    t_last_nz = float(tt_inf[i_last])

    h2_val = np.nan
    if nh2 is not None and len(np.ravel(nh2)) > 0:
        y_h2 = np.ravel(np.real(nh2)).astype(float)
        has_time_h2 = t_h2 is not None and len(np.ravel(t_h2)) == len(y_h2)
        if has_time_h2:
            tt_h2 = np.ravel(t_h2).astype(float)
            idx_h2 = np.where(np.isclose(tt_h2, t_last_nz))[0]
            if idx_h2.size > 0:
                h2_val = float(y_h2[int(idx_h2[0])])
            else:
                j = int(np.argmin(np.abs(tt_h2 - t_last_nz)))
                h2_val = float(y_h2[j])
        else:
            if i_last < len(y_h2):
                h2_val = float(y_h2[i_last])

    print(f"--- {label} ---")
    print(f"t_last_nonzero_norm_inf = {sci_notation(t_last_nz, n_decimals=4)}")
    print(f"H2_at_that_time = {h2_val:.17f}")
    if not has_time_inf:
        print("warning: norm_inf_t no estaba alineado; se uso indice como tiempo")
    print()

In [ ]:
def _fwhm(x, y):
    x = np.ravel(x).astype(float)
    y = np.ravel(y).astype(float)
    if x.size < 2 or y.size < 2:
        return np.nan, (np.nan, np.nan)
    peak = float(np.max(y))
    if not np.isfinite(peak) or peak <= 0:
        return np.nan, (np.nan, np.nan)
    half = 0.5 * peak
    idx = np.where(y >= half)[0]
    if idx.size == 0:
        return np.nan, (np.nan, np.nan)
    i_left = int(idx[0])
    i_right = int(idx[-1])

    def _interp_left(i):
        if i == 0:
            return float(x[0])
        x0, x1 = float(x[i-1]), float(x[i])
        y0, y1 = float(y[i-1]), float(y[i])
        if np.isclose(y1, y0):
            return x1
        return x0 + (half - y0) * (x1 - x0) / (y1 - y0)

    def _interp_right(i):
        if i >= x.size - 1:
            return float(x[-1])
        x0, x1 = float(x[i]), float(x[i+1])
        y0, y1 = float(y[i]), float(y[i+1])
        if np.isclose(y1, y0):
            return x0
        return x0 + (half - y0) * (x1 - x0) / (y1 - y0)

    x_left = _interp_left(i_left)
    x_right = _interp_right(i_right)
    return float(x_right - x_left), (x_left, x_right)


def _rms_width(x, rho, L=None):
    x = np.ravel(x).astype(float)
    rho = np.ravel(np.real(rho)).astype(float)
    mask = np.isfinite(x) & np.isfinite(rho) & (rho >= 0)
    if L is not None:
        L = float(L)
        mask &= np.abs(x) <= L
    x = x[mask]
    rho = rho[mask]
    if x.size < 2 or rho.size < 2:
        return np.nan, np.nan, np.nan

    dx = np.diff(x)
    if dx.size == 0:
        return np.nan, np.nan, np.nan
    h = dx[0] #float(np.mean(dx))
    if not np.isfinite(h) or h <= 0:
        return np.nan, np.nan, np.nan
    if not np.allclose(dx, h, rtol=1e-8, atol=1e-12):
        warnings.warn("x no uniforme: el ancho RMS usa suma discreta con h promedio.")

    norm = h * np.sum(rho)
    if not np.isfinite(norm) or norm <= 0:
        return np.nan, np.nan, np.nan

    xbar = h * np.sum(x * rho) / norm
    var = h * np.sum((x - xbar) ** 2 * rho) / norm
    return float(np.sqrt(var)), float(xbar), float(norm)


L_RMS = 3 #0.0001

for run_key, d in runs.items():
    label = "Not refined" if "no_refine" in run_key.lower() else "Refined"
    x = d.get("x_plot")
    u = d.get("uxx_last")
    if x is None or u is None or len(np.ravel(u)) == 0:
        print(f"--- {label} ---")
        print("No hay u_last/x_plot disponibles")
        print()
        continue

    x = np.ravel(x).astype(float)
    u = np.ravel(u)
    mod = np.abs(u)

    fwhm_val, (xl, xr) = _fwhm(x, mod)
    rms_mod, xbar_mod, norm_mod = _rms_width(x, mod, L=L_RMS)

    print(f"--- {label} ---")
    print(f"max|u_xx| = {format(float(np.max(mod)), '.17f')}")
    print(f"FWHM(|u_xx|) = {format(float(fwhm_val), '.17f')}")
    print(f"RMS width of |u_xx| on [-L, L], L={format(float(L_RMS), '.17f')} = {format(float(rms_mod), '.17f')}   center = {format(float(xbar_mod), '.17f')}")
    print(f"Normalization on [-L, L] = {format(float(norm_mod), '.17f')}")
    print(f"FWHM interval for |u_xx| = [{format(float(xl), '.17f')}, {format(float(xr), '.17f')}]")
    print()

In [ ]:
def _moving_average(y, window):
    y = np.ravel(np.real(y)).astype(float)
    window = int(window)
    if window <= 1 or y.size < window:
        return y
    kernel = np.ones(window, dtype=float) / float(window)
    return np.convolve(y, kernel, mode="same")


def _first_positive_local_minimum(x, y, smooth_window=20, min_relative_drop=0.1):
    x = np.ravel(x).astype(float)
    y = np.ravel(np.real(y)).astype(float)

    mask = np.isfinite(x) & np.isfinite(y) & (x > 0)
    x_pos = x[mask]
    y_pos = y[mask]

    if x_pos.size < 3:
        return np.nan, np.nan, None

    y_smooth = _moving_average(y_pos, smooth_window)
    baseline = float(np.max(y_smooth)) if y_smooth.size else np.nan
    if not np.isfinite(baseline) or baseline <= 0:
        return np.nan, np.nan, None

    candidates = []
    for i in range(1, x_pos.size - 1):
        left = y_smooth[i - 1]
        mid = y_smooth[i]
        right = y_smooth[i + 1]
        if mid <= left and mid <= right:
            if (baseline - mid) / baseline >= float(min_relative_drop):
                candidates.append(i)

    if candidates:
        i = int(candidates[0])
    else:
        i = int(np.argmin(y_smooth))

    return float(x_pos[i]), float(y_pos[i]), i


def _quadratic_minimum_from_seed(x, y, seed_x, center_idx, half_window=100):
    x = np.ravel(x).astype(float)
    y = np.ravel(np.real(y)).astype(float)
    if center_idx is None or x.size < 3 or y.size < 3:
        return np.nan, np.nan, None

    seed_x = float(seed_x)
    center_idx = int(center_idx)
    half_window = int(half_window)

    left = max(0, center_idx - half_window)
    right = min(x.size, center_idx + half_window + 1)
    x_win = x[left:right]
    y_win = y[left:right]
    mask = np.isfinite(x_win) & np.isfinite(y_win) & (x_win > 0)
    x_win = x_win[mask]
    y_win = y_win[mask]

    if x_win.size < 3:
        return np.nan, np.nan, None

    x_shift = x_win - seed_x
    coeffs = np.polyfit(x_shift, y_win, deg=2)
    a2, a1, a0 = map(float, coeffs)
    if (not np.isfinite(a2)) or np.isclose(a2, 0.0):
        i_local = int(np.argmin(y_win))
        return float(x_win[i_local]), float(y_win[i_local]), left + i_local

    shift_vertex = -a1 / (2.0 * a2)
    x_vertex = seed_x + shift_vertex
    y_vertex = a2 * shift_vertex**2 + a1 * shift_vertex + a0

    dx_med = float(np.median(np.diff(x_win))) if x_win.size >= 2 else np.nan
    max_shift = 5.0 * abs(dx_med) if np.isfinite(dx_med) and dx_med > 0 else np.inf

    if (not np.isfinite(x_vertex)) or (not np.isfinite(y_vertex)) or (x_vertex <= 0.0):
        i_local = int(np.argmin(y_win))
        return float(x_win[i_local]), float(y_win[i_local]), left + i_local

    if abs(shift_vertex) > max_shift:
        i_local = int(np.argmin(y_win))
        return float(x_win[i_local]), float(y_win[i_local]), left + i_local

    if x_vertex < float(np.min(x_win)) or x_vertex > float(np.max(x_win)):
        i_local = int(np.argmin(y_win))
        return float(x_win[i_local]), float(y_win[i_local]), left + i_local

    return float(x_vertex), float(y_vertex), center_idx


ROUGH_SMOOTH_WINDOW = 80
ROUGH_MIN_RELATIVE_DROP = 0.01 #0.1
FIT_HALF_WINDOW = 10_000
FIT_HALF_WINDOW_NO_REFINE = max(1, FIT_HALF_WINDOW // 10)

rough_minima = {}
for run_key, d in runs.items():
    x = d.get("x_plot")
    uxx = d.get("uxx_last")

    if x is None or uxx is None or len(np.ravel(uxx)) == 0:
        rough_minima[run_key] = {
            "x_pos": None,
            "y_pos": None,
            "x_seed": np.nan,
            "uxx_seed": np.nan,
            "idx_seed": None,
        }
        continue

    x0 = np.ravel(x).astype(float)
    abs_uxx = np.abs(np.ravel(uxx))
    mask_pos = np.isfinite(x0) & np.isfinite(abs_uxx) & (x0 > 0)
    x_pos = x0[mask_pos]
    y_pos = abs_uxx[mask_pos]
    x_seed, uxx_seed, idx_seed = _first_positive_local_minimum(
        x_pos,
        y_pos,
        smooth_window=ROUGH_SMOOTH_WINDOW,
        min_relative_drop=ROUGH_MIN_RELATIVE_DROP,
    )
    rough_minima[run_key] = {
        "x_pos": x_pos,
        "y_pos": y_pos,
        "x_seed": x_seed,
        "uxx_seed": uxx_seed,
        "idx_seed": idx_seed,
    }

minima_results = {}
reference_key = next((k for k in rough_minima if _is_no_refine_key(k)), None)
reference_x = rough_minima[reference_key]["x_seed"] if reference_key is not None else np.nan
reference_uxx = rough_minima[reference_key]["uxx_seed"] if reference_key is not None else np.nan

for run_key, d in runs.items():
    label = "Not refined" if _is_no_refine_key(run_key) else "Refined"
    data = rough_minima.get(run_key, {})
    x_pos = data.get("x_pos")
    y_pos = data.get("y_pos")
    x_seed = data.get("x_seed", np.nan)
    uxx_seed = data.get("uxx_seed", np.nan)
    idx_seed = data.get("idx_seed")

    if x_pos is None or y_pos is None or len(x_pos) == 0:
        print(f"--- {label} ---")
        print("No hay x_plot/uxx_last disponibles")
        print()
        continue

    if idx_seed is None:
        x_min, uxx_min, idx_min = np.nan, np.nan, None
    else:
        half_window = FIT_HALF_WINDOW_NO_REFINE if _is_no_refine_key(run_key) else FIT_HALF_WINDOW
        x_min, uxx_min, idx_min = _quadratic_minimum_from_seed(
            x_pos,
            y_pos,
            x_seed,
            idx_seed,
            half_window=half_window,
        )

    delta_ref = abs(float(x_min) - float(reference_x)) if (np.isfinite(x_min) and np.isfinite(reference_x)) else np.nan

    print(f"--- {label} ---")
    if idx_min is None:
        print("No se encontro un minimo local positivo")
    else:
        print(f"semilla de la primera version = {format(float(x_seed), '.17f')}")
        print(f"|u_xx| en la semilla = {format(float(uxx_seed), '.17f')}")
        print(f"minimo cuadratico refinado = {format(float(x_min), '.17f')}")
        print(f"|u_xx|(x) = {format(float(uxx_min), '.17f')}")
        if not _is_no_refine_key(run_key):
            print(f"referencia no refinada = {format(float(reference_x), '.17f')}")
            print(f"|delta x| vs no refinado = {format(float(delta_ref), '.17f')}")
    minima_results[run_key] = {
        "label": label,
        "x_min": x_min,
        "uxx_min": uxx_min,
        "x_seed": x_seed,
        "uxx_seed": uxx_seed,
    }
    print()
reference_key = next((k for k in minima_results if _is_no_refine_key(k)), None)
reference_x = minima_results[reference_key]["x_min"] if reference_key is not None else np.nan
reference_uxx = minima_results[reference_key]["uxx_min"] if reference_key is not None else np.nan

# H2 refinada empalmada para varios A


In [ ]:
AS_H2_REFINED = [0.638, 0.644, 0.65, 0.656]
A_BATCH = {
    0.638: "second",
    0.644: "second",
    0.65: "third",
    0.656: "third",
}
AS_ROOT = Path(
    "../../../scripts/Results/Blow_up/As"
)

def _a_folder_name(A):
    return f"blow_up_A_{A:g}"

def _load_real_array(path):
    return np.ravel(np.real(load_npy_auto(Path(path)))).astype(float)

def _trim_to_last_nonzero(t, y, zero_tol=1e-14):
    t = np.ravel(t).astype(float)
    y = np.ravel(np.real(y)).astype(float)
    n = min(len(t), len(y))
    t = t[:n]
    y = y[:n]
    idx = np.where(np.isfinite(y) & (np.abs(y) > zero_tol))[0]
    if idx.size == 0:
        return np.array([], dtype=float), np.array([], dtype=float)
    last = int(idx[-1])
    return t[:last + 1], y[:last + 1]

def _find_refined_restart_dir(a_dir):
    restart_dir = Path(a_dir) / "restart_runs"
    if not restart_dir.exists():
        raise FileNotFoundError(f"No existe restart_runs en {a_dir}")

    candidates = []
    for d in sorted(restart_dir.iterdir()):
        if not d.is_dir():
            continue
        name = d.name.lower()
        is_refined = (
            ("dx_10x" in name and "dt_10x" in name)
            or ("10dx" in name and "10dt" in name)
        )
        if is_refined and "no_refine" not in name:
            candidates.append(d)

    if not candidates:
        raise FileNotFoundError(f"No encontre corrida refinada 10x en {restart_dir}")
    if len(candidates) > 1:
        warnings.warn(f"Hay varias corridas refinadas en {restart_dir}; uso {candidates[0].name}")
    return candidates[0]

def _restart_start_index(restart_run_dir):
    m = re.search(r"_from_(\d+)", Path(restart_run_dir).name)
    return int(m.group(1)) if m else None

def _align_restart_time(t_base, t_ref, restart_run_dir):
    t_base = np.ravel(t_base).astype(float)
    t_ref = np.ravel(t_ref).astype(float)
    if t_base.size == 0 or t_ref.size == 0:
        return t_ref

    # Si el tiempo refinado ya parece absoluto, lo dejamos tal cual.
    if t_ref[0] > t_base[0] and t_ref[-1] <= t_base[-1] * 1.05:
        return t_ref

    start_idx = _restart_start_index(restart_run_dir)
    if start_idx is None:
        warnings.warn(f"No pude leer indice inicial de {Path(restart_run_dir).name}; uso tiempo refinado sin desplazar.")
        return t_ref

    k = min(max(start_idx - 1, 0), len(t_base) - 1)
    return t_ref + (t_base[k] - t_ref[0])

def load_base_plus_refined_h2(A):
    batch = A_BATCH[float(A)]
    a_dir = AS_ROOT / f"blow_up_N_160000_Nt_64221_sigma_0.1_{batch}" / _a_folder_name(A)
    refined_dir = _find_refined_restart_dir(a_dir)

    t_base, h2_base = _trim_to_last_nonzero(
        _load_real_array(a_dir / "time.npy.gz"),
        _load_real_array(a_dir / "H2_norm.npy.gz"),
    )
    t_ref, h2_ref = _trim_to_last_nonzero(
        _load_real_array(refined_dir / "time.npy.gz"),
        _load_real_array(refined_dir / "H2_norm.npy.gz"),
    )
    t_ref = _align_restart_time(t_base, t_ref, refined_dir)

    if t_ref.size == 0:
        t_all, h2_all = t_base, h2_base
    else:
        prefix = t_base < t_ref[0]
        t_all = np.concatenate([t_base[prefix], t_ref])
        h2_all = np.concatenate([h2_base[prefix], h2_ref])

    order = np.argsort(t_all)
    t_all = t_all[order]
    h2_all = h2_all[order]

    keep = np.isfinite(t_all) & np.isfinite(h2_all) & (h2_all > 0)
    t_all = t_all[keep]
    h2_all = h2_all[keep]

    if t_all.size == 0:
        raise ValueError(f"Serie H2 vacia para A={A}")

    T = float(t_all[-1])
    return {
        "A": float(A),
        "a_dir": a_dir,
        "refined_dir": refined_dir,
        "t": t_all,
        "H2": h2_all,
        "T": T,
        "n_base_prefix": int(np.sum(t_base < t_ref[0])) if t_ref.size else len(t_base),
        "n_refined": int(len(t_ref)),
    }

h2_refined_by_A = {A: load_base_plus_refined_h2(A) for A in AS_H2_REFINED}

for A, data in h2_refined_by_A.items():
    print(
        f"A={A:g}: T={data['T']:.17e}, "
        f"base_prefix={data['n_base_prefix']}, refined={data['n_refined']}, "
        f"refined_dir={data['refined_dir'].name}"
    )


In [ ]:
STEPS_MIN_BEFORE_T = 0
STEPS_MAX_BEFORE_T = 5_000
SHOW_ALL_POINTS = False
DISPLAY_MAX_POINTS = 40
REFINE_START_MARKER = "*"
REFINE_START_MARKER_COLOR = "black"

try:
    from labellines import labelLines
except ImportError:
    labelLines = None

plt.figure(figsize=(4, 3)) #7, 3
fit_lines = []
fit_xvals = []
fit_label_points = []

def _display_sample(x, y, max_points=DISPLAY_MAX_POINTS):
    x = np.ravel(x)
    y = np.ravel(y)
    if SHOW_ALL_POINTS or len(x) <= max_points:
        return x, y

    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]

    x_targets = np.linspace(float(np.min(x_sorted)), float(np.max(x_sorted)), int(max_points))
    idx = np.searchsorted(x_sorted, x_targets)
    idx = np.clip(idx, 0, len(x_sorted) - 1)
    idx = np.unique(idx)

    return x_sorted[idx], y_sorted[idx]

def _original_h2_log_plot_data(A):
    batch = A_BATCH[float(A)]
    a_dir = AS_ROOT / f"blow_up_N_160000_Nt_64221_sigma_0.1_{batch}" / _a_folder_name(A)
    time_full = _load_real_array(a_dir / "time.npy.gz")
    h2_full = _load_real_array(a_dir / "H2_norm.npy.gz")
    norm_inf_full = _load_real_array(a_dir / "norm_inf.npy.gz")

    t_base, h2_base = _trim_to_last_nonzero(time_full, h2_full)

    n_inf = min(len(time_full), len(norm_inf_full))
    t_inf = np.ravel(time_full[:n_inf]).astype(float)
    norm_inf = np.ravel(norm_inf_full[:n_inf]).astype(float)
    inf_mask = np.isfinite(t_inf) & np.isfinite(norm_inf) & (np.abs(norm_inf) > 1e-14)
    below_target = np.where(inf_mask & (norm_inf < TARGET_INF))[0]
    if below_target.size == 0:
        return np.array([], dtype=float), np.array([], dtype=float)
    T_base = float(t_inf[int(below_target[-1])])

    keep = np.isfinite(t_base) & np.isfinite(h2_base) & (h2_base > 0) & (t_base <= T_base)
    t_base = t_base[keep]
    h2_base = h2_base[keep]

    dt_vals = np.diff(t_base)
    dt_vals = dt_vals[np.isfinite(dt_vals) & (dt_vals > 0)]
    dt = float(np.median(dt_vals)) if dt_vals.size else np.nan

    tau = T_base - t_base
    mask = np.isfinite(tau) & np.isfinite(h2_base) & (tau > 0) & (h2_base > 0) & (-np.log(tau) > 0)
    if np.isfinite(dt):
        tau_min = STEPS_MIN_BEFORE_T * dt
        tau_max = STEPS_MAX_BEFORE_T * dt
        mask &= (tau >= tau_min) & (tau <= tau_max)

    return np.log(-np.log(tau[mask])), np.log(h2_base[mask])

prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()
colors = prop_cycle["color"]
markers = prop_cycle.get("marker", [None])

for j, (A, data) in enumerate(h2_refined_by_A.items()):
    color = colors[j % len(colors)]
    marker = markers[j % len(markers)]
    t = data["t"]
    h2 = data["H2"]
    T = data["T"]

    dt_vals = np.diff(t)
    dt_vals = dt_vals[np.isfinite(dt_vals) & (dt_vals > 0)]
    dt = float(np.median(dt_vals)) if dt_vals.size else np.nan

    tau = T - t
    mask = np.isfinite(tau) & np.isfinite(h2) & (tau > 0) & (h2 > 0) & (-np.log(tau) > 0)
    if np.isfinite(dt):
        tau_min = STEPS_MIN_BEFORE_T * dt
        tau_max = STEPS_MAX_BEFORE_T * dt
        mask &= (tau >= tau_min) & (tau <= tau_max)

    x_plot = np.log(-np.log(tau[mask]))
    y_plot = np.log(h2[mask])

    x_base_plot, y_base_plot = _original_h2_log_plot_data(A)
    if len(x_base_plot) > 0:
        x_base_show, y_base_show = _display_sample(x_base_plot, y_base_plot)
        plt.plot(
            x_base_show,
            y_base_show,
            color=color,
            alpha=0.85,
            ms=3,
            label="_nolegend_",
            zorder=1,
            marker=marker,
            ls="-",
        )

    x_show, y_show = _display_sample(x_plot, y_plot)
    plt.plot(x_show, y_show, color=color, label=rf"$A={A:g}$", ms=4,ls="",
             zorder=2, markeredgewidth=.6, marker=marker, markerfacecolor="none")

    ref_idx = data.get("n_base_prefix")
    if ref_idx is not None and 0 <= ref_idx < len(t):
        tau_ref = T - t[int(ref_idx)]
        h2_ref = h2[int(ref_idx)]
        if np.isfinite(tau_ref) and np.isfinite(h2_ref) and tau_ref > 0 and h2_ref > 0 and -np.log(tau_ref) > 0:
            x_ref = np.log(-np.log(tau_ref))
            y_ref = np.log(h2_ref)
            if (len(x_plot) == 0) or (np.min(x_plot) <= x_ref <= np.max(x_plot)):
                plt.plot(
                    x_ref,
                    y_ref,
                    marker=REFINE_START_MARKER,
                    color=REFINE_START_MARKER_COLOR,
                    ms=7,
                    markeredgewidth=1,
                    ls="",
                    label="_nolegend_",
                    zorder=4,
                )

    if len(x_plot) >= 2:
        slope, intercept = np.polyfit(x_plot, y_plot, 1)
        y_fit = slope * x_plot + intercept
        fit_line, = plt.plot(
            x_plot,
            y_fit,
            "--",
            lw=1.2,
            color="black",
            alpha=0.85,
            label=rf"$p={slope:.4f}$",
            zorder=3,
        )
        fit_lines.append(fit_line)
        x_label = float(np.quantile(x_plot, 0.72))
        fit_xvals.append(x_label)
        fit_label_points.append((x_label, float(slope * x_label + intercept), rf"$p={slope:.4f}$"))
        print(f"A={A:g}: pendiente = {slope:.17f}, dt = {dt:.17e}")
    else:
        print(f"A={A:g}: no hay suficientes puntos para la regresion con los pasos dados")

if fit_lines and labelLines is not None:
    labelLines(
        fit_lines,
        xvals=2.55,#fit_xvals,
        fontsize=7,
        backgroundcolor="none",
        drop_label=True,
        zorder=2.5,
        outline_width=0,
        yoffsets=-0.1
    )
elif fit_label_points:
    for x_label, y_label, text in fit_label_points:
        plt.text(x_label, y_label, text, fontsize=8, color="black")

plt.xlabel(r"$\mathrm{ln}(-\mathrm{ln}(T_{\tau, h}^*-t_n))$")
plt.ylabel(r"$\mathrm{ln}(|\Psi|_{H^2_h})$", rotation=0, labelpad=25)
plt.legend(fontsize=7)
plt.tight_layout()

plt.savefig("Img/Heaven/As/h2_log_log_plot.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
STEPS_MIN_BEFORE_T = 0
STEPS_MAX_BEFORE_T = 5_000
SHOW_ALL_POINTS = False
DISPLAY_MAX_POINTS = 40
REFINE_START_MARKER = "*"
REFINE_START_MARKER_COLOR = "black"

# D(t) := inf_x (1 + 2*kappa*|psi(x,t)|^2) = 1 + 2*kappa*||psi(t)||_{Linf}^2
# Para el blow-up defocusing, kappa = -1: D -> 0^+ cuando ||psi||_inf -> sqrt(2)/2 (= TARGET_INF)
KAPPA = -1


def _D_of_norm_inf(norm_inf):
    return 1.0 + 2.0 * KAPPA * np.asarray(norm_inf, dtype=float) ** 2

try:
    from labellines import labelLines
except ImportError:
    labelLines = None

plt.figure(figsize=(4, 3)) #7, 3
fit_lines = []
fit_xvals = []
fit_label_points = []

def _display_sample(x, y, max_points=DISPLAY_MAX_POINTS):
    x = np.ravel(x)
    y = np.ravel(y)
    if SHOW_ALL_POINTS or len(x) <= max_points:
        return x, y

    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]

    x_targets = np.linspace(float(np.min(x_sorted)), float(np.max(x_sorted)), int(max_points))
    idx = np.searchsorted(x_sorted, x_targets)
    idx = np.clip(idx, 0, len(x_sorted) - 1)
    idx = np.unique(idx)

    return x_sorted[idx], y_sorted[idx]

def _original_h2_log_plot_data(A):
    batch = A_BATCH[float(A)]
    a_dir = AS_ROOT / f"blow_up_N_160000_Nt_64221_sigma_0.1_{batch}" / _a_folder_name(A)
    time_full = _load_real_array(a_dir / "time.npy.gz")
    h2_full = _load_real_array(a_dir / "H2_norm.npy.gz")
    norm_inf_full = _load_real_array(a_dir / "norm_inf.npy.gz")

    t_base, h2_base = _trim_to_last_nonzero(time_full, h2_full)

    n_inf = min(len(time_full), len(norm_inf_full))
    t_inf = np.ravel(time_full[:n_inf]).astype(float)
    norm_inf = np.ravel(norm_inf_full[:n_inf]).astype(float)
    inf_mask = np.isfinite(t_inf) & np.isfinite(norm_inf) & (np.abs(norm_inf) > 1e-14)
    below_target = np.where(inf_mask & (norm_inf < TARGET_INF))[0]
    if below_target.size == 0:
        return np.array([], dtype=float), np.array([], dtype=float)
    T_base = float(t_inf[int(below_target[-1])])

    keep = np.isfinite(t_base) & np.isfinite(h2_base) & (h2_base > 0) & (t_base <= T_base)
    t_base = t_base[keep]
    h2_base = h2_base[keep]

    # norm_inf interpolado a los tiempos de la serie base -> D(t)
    ni_base = np.interp(t_base, t_inf[inf_mask], norm_inf[inf_mask])
    D_base = _D_of_norm_inf(ni_base)

    dt_vals = np.diff(t_base)
    dt_vals = dt_vals[np.isfinite(dt_vals) & (dt_vals > 0)]
    dt = float(np.median(dt_vals)) if dt_vals.size else np.nan

    tau = T_base - t_base
    mask = np.isfinite(tau) & np.isfinite(h2_base) & (tau > 0) & (h2_base > 0) & (D_base > 0)
    if np.isfinite(dt):
        tau_min = STEPS_MIN_BEFORE_T * dt
        tau_max = STEPS_MAX_BEFORE_T * dt
        mask &= (tau >= tau_min) & (tau <= tau_max)

    return np.log(D_base[mask]), np.log(h2_base[mask])

def load_base_plus_refined_norm_inf(A):
    """norm_inf (base + refinada) alineada en tiempo igual que H2, ordenada por t."""
    batch = A_BATCH[float(A)]
    a_dir = AS_ROOT / f"blow_up_N_160000_Nt_64221_sigma_0.1_{batch}" / _a_folder_name(A)
    refined_dir = _find_refined_restart_dir(a_dir)

    t_base, ni_base = _trim_to_last_nonzero(
        _load_real_array(a_dir / "time.npy.gz"),
        _load_real_array(a_dir / "norm_inf.npy.gz"),
    )
    t_ref, ni_ref = _trim_to_last_nonzero(
        _load_real_array(refined_dir / "time.npy.gz"),
        _load_real_array(refined_dir / "norm_inf.npy.gz"),
    )
    t_ref = _align_restart_time(t_base, t_ref, refined_dir)

    if t_ref.size == 0:
        t_all, ni_all = t_base, ni_base
    else:
        prefix = t_base < t_ref[0]
        t_all = np.concatenate([t_base[prefix], t_ref])
        ni_all = np.concatenate([ni_base[prefix], ni_ref])

    order = np.argsort(t_all)
    return t_all[order], ni_all[order]

prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()
colors = prop_cycle["color"]
markers = prop_cycle.get("marker", [None])

for j, (A, data) in enumerate(h2_refined_by_A.items()):
    color = colors[j % len(colors)]
    marker = markers[j % len(markers)]
    t = data["t"]
    h2 = data["H2"]
    T = data["T"]

    t_ni, ni_all = load_base_plus_refined_norm_inf(A)
    norm_inf_t = np.interp(t, t_ni, ni_all)
    D_t = _D_of_norm_inf(norm_inf_t)

    dt_vals = np.diff(t)
    dt_vals = dt_vals[np.isfinite(dt_vals) & (dt_vals > 0)]
    dt = float(np.median(dt_vals)) if dt_vals.size else np.nan

    tau = T - t
    mask = np.isfinite(tau) & np.isfinite(h2) & (tau > 0) & (h2 > 0) & (D_t > 0)
    if np.isfinite(dt):
        tau_min = STEPS_MIN_BEFORE_T * dt
        tau_max = STEPS_MAX_BEFORE_T * dt
        mask &= (tau >= tau_min) & (tau <= tau_max)

    x_plot = np.log(D_t[mask])
    y_plot = np.log(h2[mask])

    x_base_plot, y_base_plot = _original_h2_log_plot_data(A)
    if len(x_base_plot) > 0:
        x_base_show, y_base_show = _display_sample(x_base_plot, y_base_plot)
        plt.plot(
            x_base_show,
            y_base_show,
            color=color,
            alpha=0.85,
            ms=3,
            label="_nolegend_",
            zorder=1,
            marker=marker,
            ls="-",
        )

    x_show, y_show = _display_sample(x_plot, y_plot)
    plt.plot(x_show, y_show, color=color, label=rf"$A={A:g}$", ms=4,ls="",
             zorder=2, markeredgewidth=.6, marker=marker, markerfacecolor="none")

    ref_idx = data.get("n_base_prefix")
    if ref_idx is not None and 0 <= ref_idx < len(t):
        D_ref = float(_D_of_norm_inf(np.interp(t[int(ref_idx)], t_ni, ni_all)))
        h2_ref = h2[int(ref_idx)]
        if np.isfinite(D_ref) and np.isfinite(h2_ref) and D_ref > 0 and h2_ref > 0:
            x_ref = np.log(D_ref)
            y_ref = np.log(h2_ref)
            if (len(x_plot) == 0) or (np.min(x_plot) <= x_ref <= np.max(x_plot)):
                plt.plot(
                    x_ref,
                    y_ref,
                    marker=REFINE_START_MARKER,
                    color=REFINE_START_MARKER_COLOR,
                    ms=7,
                    markeredgewidth=1,
                    ls="",
                    label="_nolegend_",
                    zorder=4,
                )

    if len(x_plot) >= 2:
        slope, intercept = np.polyfit(x_plot, y_plot, 1)
        y_fit = slope * x_plot + intercept
        fit_line, = plt.plot(
            x_plot,
            y_fit,
            "--",
            lw=1.2,
            color="black",
            alpha=0.85,
            label=rf"$p={slope:.4f}$",
            zorder=3,
        )
        fit_lines.append(fit_line)
        x_label = float(np.quantile(x_plot, 0.72))
        fit_xvals.append(x_label)
        fit_label_points.append((x_label, float(slope * x_label + intercept), rf"$p={slope:.4f}$"))
        print(f"A={A:g}: pendiente = {slope:.17f}, dt = {dt:.17e}")
    else:
        print(f"A={A:g}: no hay suficientes puntos para la regresion con los pasos dados")

if fit_lines and labelLines is not None:
    labelLines(
        fit_lines,
        xvals=fit_xvals,
        fontsize=7,
        backgroundcolor="none",
        drop_label=True,
        zorder=2.5,
        outline_width=0,
        yoffsets=-0.1
    )
elif fit_label_points:
    for x_label, y_label, text in fit_label_points:
        plt.text(x_label, y_label, text, fontsize=8, color="black")

plt.xlabel(r"$\mathrm{ln}(\mathcal{D}(t_n))$")
plt.ylabel(r"$\mathrm{ln}(|\Psi|_{H^2_h})$", rotation=0, labelpad=25)
plt.legend(fontsize=7)
plt.tight_layout()

plt.savefig("Img/Heaven/As/h2_log_log_plot_D.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
STEPS_MIN_BEFORE_T = 0
STEPS_MAX_BEFORE_T = 5_000
SHOW_ALL_POINTS = False
DISPLAY_MAX_POINTS = 40
REFINE_START_MARKER = "*"
REFINE_START_MARKER_COLOR = "black"

# D(t) := inf_x (1 + 2*kappa*|psi(x,t)|^2) = 1 + 2*kappa*||psi(t)||_{Linf}^2
# Para el blow-up defocusing, kappa = -1: D -> 0^+ cuando ||psi||_inf -> sqrt(2)/2 (= TARGET_INF)
KAPPA = -1


def _D_of_norm_inf(norm_inf):
    return 1.0 + 2.0 * KAPPA * np.asarray(norm_inf, dtype=float) ** 2

try:
    from labellines import labelLines
except ImportError:
    labelLines = None

plt.figure(figsize=(4, 3)) #7, 3
fit_lines = []
fit_xvals = []
fit_label_points = []

def _display_sample(x, y, max_points=DISPLAY_MAX_POINTS):
    x = np.ravel(x)
    y = np.ravel(y)
    if SHOW_ALL_POINTS or len(x) <= max_points:
        return x, y

    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]

    x_targets = np.linspace(float(np.min(x_sorted)), float(np.max(x_sorted)), int(max_points))
    idx = np.searchsorted(x_sorted, x_targets)
    idx = np.clip(idx, 0, len(x_sorted) - 1)
    idx = np.unique(idx)

    return x_sorted[idx], y_sorted[idx]

def _original_h2_log_plot_data(A):
    batch = A_BATCH[float(A)]
    a_dir = AS_ROOT / f"blow_up_N_160000_Nt_64221_sigma_0.1_{batch}" / _a_folder_name(A)
    time_full = _load_real_array(a_dir / "time.npy.gz")
    h2_full = _load_real_array(a_dir / "H2_norm.npy.gz")
    norm_inf_full = _load_real_array(a_dir / "norm_inf.npy.gz")

    t_base, h2_base = _trim_to_last_nonzero(time_full, h2_full)

    n_inf = min(len(time_full), len(norm_inf_full))
    t_inf = np.ravel(time_full[:n_inf]).astype(float)
    norm_inf = np.ravel(norm_inf_full[:n_inf]).astype(float)
    inf_mask = np.isfinite(t_inf) & np.isfinite(norm_inf) & (np.abs(norm_inf) > 1e-14)
    below_target = np.where(inf_mask & (norm_inf < TARGET_INF))[0]
    if below_target.size == 0:
        return np.array([], dtype=float), np.array([], dtype=float)
    T_base = float(t_inf[int(below_target[-1])])

    keep = np.isfinite(t_base) & np.isfinite(h2_base) & (h2_base > 0) & (t_base <= T_base)
    t_base = t_base[keep]
    h2_base = h2_base[keep]

    # norm_inf interpolado a los tiempos de la serie base -> D(t)
    ni_base = np.interp(t_base, t_inf[inf_mask], norm_inf[inf_mask])
    D_base = _D_of_norm_inf(ni_base)

    dt_vals = np.diff(t_base)
    dt_vals = dt_vals[np.isfinite(dt_vals) & (dt_vals > 0)]
    dt = float(np.median(dt_vals)) if dt_vals.size else np.nan

    tau = T_base - t_base
    mask = np.isfinite(tau) & np.isfinite(h2_base) & (tau > 0) & (h2_base > 0) & (D_base > 0) & (D_base < 1)
    if np.isfinite(dt):
        tau_min = STEPS_MIN_BEFORE_T * dt
        tau_max = STEPS_MAX_BEFORE_T * dt
        mask &= (tau >= tau_min) & (tau <= tau_max)

    return np.log(np.abs(np.log(D_base[mask]))), np.log(h2_base[mask])

def load_base_plus_refined_norm_inf(A):
    """norm_inf (base + refinada) alineada en tiempo igual que H2, ordenada por t."""
    batch = A_BATCH[float(A)]
    a_dir = AS_ROOT / f"blow_up_N_160000_Nt_64221_sigma_0.1_{batch}" / _a_folder_name(A)
    refined_dir = _find_refined_restart_dir(a_dir)

    t_base, ni_base = _trim_to_last_nonzero(
        _load_real_array(a_dir / "time.npy.gz"),
        _load_real_array(a_dir / "norm_inf.npy.gz"),
    )
    t_ref, ni_ref = _trim_to_last_nonzero(
        _load_real_array(refined_dir / "time.npy.gz"),
        _load_real_array(refined_dir / "norm_inf.npy.gz"),
    )
    t_ref = _align_restart_time(t_base, t_ref, refined_dir)

    if t_ref.size == 0:
        t_all, ni_all = t_base, ni_base
    else:
        prefix = t_base < t_ref[0]
        t_all = np.concatenate([t_base[prefix], t_ref])
        ni_all = np.concatenate([ni_base[prefix], ni_ref])

    order = np.argsort(t_all)
    return t_all[order], ni_all[order]

prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()
colors = prop_cycle["color"]
markers = prop_cycle.get("marker", [None])

for j, (A, data) in enumerate(h2_refined_by_A.items()):
    color = colors[j % len(colors)]
    marker = markers[j % len(markers)]
    t = data["t"]
    h2 = data["H2"]
    T = data["T"]

    t_ni, ni_all = load_base_plus_refined_norm_inf(A)
    norm_inf_t = np.interp(t, t_ni, ni_all)
    D_t = _D_of_norm_inf(norm_inf_t)

    dt_vals = np.diff(t)
    dt_vals = dt_vals[np.isfinite(dt_vals) & (dt_vals > 0)]
    dt = float(np.median(dt_vals)) if dt_vals.size else np.nan

    tau = T - t
    mask = np.isfinite(tau) & np.isfinite(h2) & (tau > 0) & (h2 > 0) & (D_t > 0) & (D_t < 1)
    if np.isfinite(dt):
        tau_min = STEPS_MIN_BEFORE_T * dt
        tau_max = STEPS_MAX_BEFORE_T * dt
        mask &= (tau >= tau_min) & (tau <= tau_max)

    x_plot = np.log(np.abs(np.log(D_t[mask])))
    y_plot = np.log(h2[mask])

    x_base_plot, y_base_plot = _original_h2_log_plot_data(A)
    if len(x_base_plot) > 0:
        x_base_show, y_base_show = _display_sample(x_base_plot, y_base_plot)
        plt.plot(
            x_base_show,
            y_base_show,
            color=color,
            alpha=0.85,
            ms=3,
            label="_nolegend_",
            zorder=1,
            marker=marker,
            ls="-",
        )

    x_show, y_show = _display_sample(x_plot, y_plot)
    plt.plot(x_show, y_show, color=color, label=rf"$A={A:g}$", ms=4,ls="",
             zorder=2, markeredgewidth=.6, marker=marker, markerfacecolor="none")

    ref_idx = data.get("n_base_prefix")
    if ref_idx is not None and 0 <= ref_idx < len(t):
        D_ref = float(_D_of_norm_inf(np.interp(t[int(ref_idx)], t_ni, ni_all)))
        h2_ref = h2[int(ref_idx)]
        if np.isfinite(D_ref) and np.isfinite(h2_ref) and D_ref > 0 and h2_ref > 0:
            x_ref = np.log(abs(np.log(D_ref)))
            y_ref = np.log(h2_ref)
            if (len(x_plot) == 0) or (np.min(x_plot) <= x_ref <= np.max(x_plot)):
                plt.plot(
                    x_ref,
                    y_ref,
                    marker=REFINE_START_MARKER,
                    color=REFINE_START_MARKER_COLOR,
                    ms=7,
                    markeredgewidth=1,
                    ls="",
                    label="_nolegend_",
                    zorder=4,
                )

    if len(x_plot) >= 2:
        slope, intercept = np.polyfit(x_plot, y_plot, 1)
        y_fit = slope * x_plot + intercept
        fit_line, = plt.plot(
            x_plot,
            y_fit,
            "--",
            lw=1.2,
            color="black",
            alpha=0.85,
            label=rf"$p={slope:.4f}$",
            zorder=3,
        )
        fit_lines.append(fit_line)
        x_label = float(np.quantile(x_plot, 0.72))
        fit_xvals.append(x_label)
        fit_label_points.append((x_label, float(slope * x_label + intercept), rf"$p={slope:.4f}$"))
        print(f"A={A:g}: pendiente = {slope:.17f}, dt = {dt:.17e}")
    else:
        print(f"A={A:g}: no hay suficientes puntos para la regresion con los pasos dados")

if fit_lines and labelLines is not None:
    labelLines(
        fit_lines,
        xvals=fit_xvals,
        fontsize=7,
        backgroundcolor="none",
        drop_label=True,
        zorder=2.5,
        outline_width=0,
        yoffsets=-0.1
    )
elif fit_label_points:
    for x_label, y_label, text in fit_label_points:
        plt.text(x_label, y_label, text, fontsize=8, color="black")

plt.xlabel(r"$\mathrm{ln}(|\mathrm{ln}(\mathcal{D}(t_n))|)$")
plt.ylabel(r"$\mathrm{ln}(|\Psi|_{H^2_h})$", rotation=0, labelpad=25)
plt.legend(fontsize=7)
plt.tight_layout()

plt.savefig("Img/Heaven/As/h2_log_log_plot_lnD.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
STEPS_MIN_BEFORE_T = 0
STEPS_MAX_BEFORE_T = 5_000
SHOW_ALL_POINTS = False
DISPLAY_MAX_POINTS = 40
REFINE_START_MARKER = "*"
REFINE_START_MARKER_COLOR = "black"

# D(t) = 1 + 2*kappa*||psi(t)||_{Linf}^2  (kappa = -1 -> D -> 0^+ en el blow-up)
KAPPA = -1


def _D_of_norm_inf(norm_inf):
    return 1.0 + 2.0 * KAPPA * np.asarray(norm_inf, dtype=float) ** 2

try:
    from labellines import labelLines
except ImportError:
    labelLines = None

plt.figure(figsize=(4, 3)) #7, 3
fit_lines = []
fit_xvals = []
fit_label_points = []

def _display_sample(x, y, max_points=DISPLAY_MAX_POINTS):
    x = np.ravel(x)
    y = np.ravel(y)
    if SHOW_ALL_POINTS or len(x) <= max_points:
        return x, y

    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]

    x_targets = np.linspace(float(np.min(x_sorted)), float(np.max(x_sorted)), int(max_points))
    idx = np.searchsorted(x_sorted, x_targets)
    idx = np.clip(idx, 0, len(x_sorted) - 1)
    idx = np.unique(idx)

    return x_sorted[idx], y_sorted[idx]

def _original_lnlnD_plot_data(A):
    """Serie base: x = ln(T*-t),  y = ln(D(t))."""
    batch = A_BATCH[float(A)]
    a_dir = AS_ROOT / f"blow_up_N_160000_Nt_64221_sigma_0.1_{batch}" / _a_folder_name(A)
    time_full = _load_real_array(a_dir / "time.npy.gz")
    h2_full = _load_real_array(a_dir / "H2_norm.npy.gz")
    norm_inf_full = _load_real_array(a_dir / "norm_inf.npy.gz")

    t_base, h2_base = _trim_to_last_nonzero(time_full, h2_full)

    n_inf = min(len(time_full), len(norm_inf_full))
    t_inf = np.ravel(time_full[:n_inf]).astype(float)
    norm_inf = np.ravel(norm_inf_full[:n_inf]).astype(float)
    inf_mask = np.isfinite(t_inf) & np.isfinite(norm_inf) & (np.abs(norm_inf) > 1e-14)
    below_target = np.where(inf_mask & (norm_inf < TARGET_INF))[0]
    if below_target.size == 0:
        return np.array([], dtype=float), np.array([], dtype=float)
    T_base = float(t_inf[int(below_target[-1])])

    keep = np.isfinite(t_base) & np.isfinite(h2_base) & (h2_base > 0) & (t_base <= T_base)
    t_base = t_base[keep]

    ni_base = np.interp(t_base, t_inf[inf_mask], norm_inf[inf_mask])
    D_base = _D_of_norm_inf(ni_base)

    dt_vals = np.diff(t_base)
    dt_vals = dt_vals[np.isfinite(dt_vals) & (dt_vals > 0)]
    dt = float(np.median(dt_vals)) if dt_vals.size else np.nan

    tau = T_base - t_base
    mask = np.isfinite(tau) & (tau > 0) & (D_base > 0)
    if np.isfinite(dt):
        tau_min = STEPS_MIN_BEFORE_T * dt
        tau_max = STEPS_MAX_BEFORE_T * dt
        mask &= (tau >= tau_min) & (tau <= tau_max)

    return np.abs(np.log(tau[mask])), np.log(D_base[mask])

def load_base_plus_refined_norm_inf(A):
    """norm_inf (base + refinada) alineada en tiempo igual que H2, ordenada por t."""
    batch = A_BATCH[float(A)]
    a_dir = AS_ROOT / f"blow_up_N_160000_Nt_64221_sigma_0.1_{batch}" / _a_folder_name(A)
    refined_dir = _find_refined_restart_dir(a_dir)

    t_base, ni_base = _trim_to_last_nonzero(
        _load_real_array(a_dir / "time.npy.gz"),
        _load_real_array(a_dir / "norm_inf.npy.gz"),
    )
    t_ref, ni_ref = _trim_to_last_nonzero(
        _load_real_array(refined_dir / "time.npy.gz"),
        _load_real_array(refined_dir / "norm_inf.npy.gz"),
    )
    t_ref = _align_restart_time(t_base, t_ref, refined_dir)

    if t_ref.size == 0:
        t_all, ni_all = t_base, ni_base
    else:
        prefix = t_base < t_ref[0]
        t_all = np.concatenate([t_base[prefix], t_ref])
        ni_all = np.concatenate([ni_base[prefix], ni_ref])

    order = np.argsort(t_all)
    return t_all[order], ni_all[order]

prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()
colors = prop_cycle["color"]
markers = prop_cycle.get("marker", [None])

for j, (A, data) in enumerate(h2_refined_by_A.items()):
    color = colors[j % len(colors)]
    marker = markers[j % len(markers)]
    t = data["t"]
    T = data["T"]

    t_ni, ni_all = load_base_plus_refined_norm_inf(A)
    norm_inf_t = np.interp(t, t_ni, ni_all)
    D_t = _D_of_norm_inf(norm_inf_t)

    dt_vals = np.diff(t)
    dt_vals = dt_vals[np.isfinite(dt_vals) & (dt_vals > 0)]
    dt = float(np.median(dt_vals)) if dt_vals.size else np.nan

    tau = T - t
    mask = np.isfinite(tau) & (tau > 0) & (D_t > 0)
    if np.isfinite(dt):
        tau_min = STEPS_MIN_BEFORE_T * dt
        tau_max = STEPS_MAX_BEFORE_T * dt
        mask &= (tau >= tau_min) & (tau <= tau_max)

    x_plot = np.abs(np.log(tau[mask]))
    y_plot = np.log(D_t[mask])

    x_base_plot, y_base_plot = _original_lnlnD_plot_data(A)
    if False: #len(x_base_plot) > 0:
        x_base_show, y_base_show = _display_sample(x_base_plot, y_base_plot)
        plt.plot(
            x_base_show,
            y_base_show,
            color=color,
            alpha=0.85,
            ms=3,
            label="_nolegend_",
            zorder=1,
            marker=marker,
            ls="-",
        )

    x_show, y_show = _display_sample(x_plot, y_plot)
    plt.plot(x_show, y_show, color=color, label=rf"$A={A:g}$", ms=4,ls="",
             zorder=2, markeredgewidth=.6, marker=marker, markerfacecolor="none")

    ref_idx = data.get("n_base_prefix")
    if ref_idx is not None and 0 <= ref_idx < len(t):
        tau_ref = T - t[int(ref_idx)]
        D_ref = float(_D_of_norm_inf(np.interp(t[int(ref_idx)], t_ni, ni_all)))
        if np.isfinite(tau_ref) and tau_ref > 0 and D_ref > 0:
            x_ref = np.log(tau_ref)
            y_ref = np.log(D_ref)
            if (len(x_plot) == 0) or (np.min(x_plot) <= x_ref <= np.max(x_plot)):
                plt.plot(
                    x_ref,
                    y_ref,
                    marker=REFINE_START_MARKER,
                    color=REFINE_START_MARKER_COLOR,
                    ms=7,
                    markeredgewidth=1,
                    ls="",
                    label="_nolegend_",
                    zorder=4,
                )

    if len(x_plot) >= 2:
        slope, intercept = np.polyfit(x_plot, y_plot, 1)
        print(f"A={A:g}: pendiente = {slope:.17f}, dt = {dt:.17e}, intercept = {intercept:.17f}")
    else:
        print(f"A={A:g}: no hay suficientes puntos para la regresion con los pasos dados")

plt.xlabel(r"$|\mathrm{ln}(T_{\tau, h}^*-t_n)|$")
plt.ylabel(r"$\mathrm{ln}(\mathcal{D}(t_n))$", rotation="horizontal", labelpad=8)
plt.legend(fontsize=7)
plt.tight_layout()

plt.savefig("Img/Heaven/As/lnD_vs_lntau.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
np.log(tau), tau

In [ ]:
np.log(T)